# GraphMS v3.5.1 — FINAL HYBRID STAGE 12 HANDOFF

**Purpose:** rebuild the frozen 93 development masks from the already-computed GraphMS v3.5.1 Hybrid probability files, reproduce the frozen cross-fitted Stage11-v1 segmentation score, and then regenerate **Stage 12 — Lesion Feature Extraction** with the existing audited `GraphMSNet-Stage12-MAX-v2.1` engine.

The attached guide places **Stage 12 after post-processing** and requires lesion **volume**, **count**, shape descriptors (**area, perimeter, compactness**) and **GLCM texture**. Stages 13–16 are downstream; this notebook intentionally **stops after Stage 12 auditing and Stage13-ready table construction**. It does **not** fit any Stage13 model.

### Frozen upstream lock
- Model: **GraphMS v3.5.1 Hybrid**
- Protocol SHA: `8944f1a0deef8a7d0eb57118b4b45e3eed0bc803a2c0e93d63d9c9eaba6578a3`
- Fusion implementation ID: `v3.5-concat-se-self-multiscale-dice-bce-adamw-cosine-250iter`
- Stored probabilities: original **uniform-overlap Hybrid** inference; **no new neural inference** is run here.
- Stage11-v1: F0 `.40/10`, F1 `.40/10`, F2 `.45/10`, F3 `.40/10`, F4 `.40/10`, 26-connectivity, no morphology.
- Frozen development result: **5-fold equal-fold DSC = 0.748049553870**.
- Claim scope: **development five-fold CV**; this is not an untouched external test.

### Scientific preservation rule
The canonical v2.1 Stage12 feature-engine functions are carried forward from the supplied source notebook. Their feature mathematics are not redesigned. The only source-engine edit is the required frozen-segmentation provenance string; runtime/cache wiring is changed only to point at the new Hybrid Stage12 namespace.


In [ ]:
#@title 0. Environment, paths, and immutable protocol locks { display-mode: "form" }
from pathlib import Path
import os, sys, json, math, re, time, hashlib, warnings, subprocess, shutil, copy
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

def pip_if_missing(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, name in [
    ('nibabel','nibabel'),
    ('scipy','scipy'),
    ('scikit-image','skimage'),
    ('nilearn','nilearn'),
]:
    pip_if_missing(pkg, name)

import nibabel as nib
from scipy import ndimage, stats
from skimage import measure

PROJECT = Path('/content/drive/MyDrive/MSLesSeg_MS')
ROOT = PROJECT / 'nnunet_v2'
RAW = ROOT / 'nnUNet_raw' / 'Dataset001_MSLesSeg'
IMAGES_TR = RAW / 'imagesTr'
IMAGES_TS = RAW / 'imagesTs'
LABELS_TR = RAW / 'labelsTr'
WINNING_RECIPE = ROOT / 'WINNING_RECIPE.json'  # retained only because the audited v2.1 cache fingerprint uses its SHA

OLD_STAGE12 = ROOT / 'stage12_canonical_features_max_v2'
OLD_STAGE12_CONTRACT = OLD_STAGE12 / 'STAGE13_RECOMMENDED_FEATURE_COLUMNS.json'
OLD_STAGE12_DEV = OLD_STAGE12 / 'stage12_case_features_dev_oof_93.csv'
OLD_STAGE13_DEV = OLD_STAGE12 / 'stage13_DEV_ONLY_93_crosssectional.csv'
OLD_ATLAS_CACHE = OLD_STAGE12 / 'atlas_cache'

OUT_ROOT = ROOT / 'graphms_hybrid_stage12_final_v1'
FINAL_MASK_DIR = OUT_ROOT / 'final_hybrid_masks'
CASE_CACHE = OUT_ROOT / 'case_cache'
REPORTS_DIR = OUT_ROOT / 'reports'
for p in [OUT_ROOT, FINAL_MASK_DIR, CASE_CACHE, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Aliases intentionally used by the unchanged canonical v2.1 source engine.
STAGE12 = OLD_STAGE12
STAGE12_DEMO_CACHE = CASE_CACHE

CLINICAL_CSV = PROJECT / 'dataset' / 'MSLesSeg Dataset' / 'info_dataset' / 'clinical_data.csv'

PROTOCOL_SHA = '8944f1a0deef8a7d0eb57118b4b45e3eed0bc803a2c0e93d63d9c9eaba6578a3'
FUSION_IMPLEMENTATION_ID = 'v3.5-concat-se-self-multiscale-dice-bce-adamw-cosine-250iter'
FEATURE_SCHEMA_VERSION_LOCK = 'GraphMSNet-Stage12-MAX-v2.1'
EXPECTED_EQUAL_FOLD_DSC = 0.748049553870
DSC_TOL = 5e-9
EXPECTED_FOLD_COUNTS = {0:22, 1:18, 2:14, 3:23, 4:16}
EXPECTED_CASES = 93
EXPECTED_PATIENTS = 53
INCLUDE_HISTORICAL_TEST = False

STAGE11_RECIPES = {
    0: {'threshold':0.40, 'minvox':10, 'connectivity':26, 'morphology':'none'},
    1: {'threshold':0.40, 'minvox':10, 'connectivity':26, 'morphology':'none'},
    2: {'threshold':0.45, 'minvox':10, 'connectivity':26, 'morphology':'none'},
    3: {'threshold':0.40, 'minvox':10, 'connectivity':26, 'morphology':'none'},
    4: {'threshold':0.40, 'minvox':10, 'connectivity':26, 'morphology':'none'},
}

required_paths = [PROJECT, ROOT, RAW, IMAGES_TR, LABELS_TR, WINNING_RECIPE,
                  OLD_STAGE12, OLD_STAGE12_CONTRACT, OLD_STAGE12_DEV,
                  OLD_STAGE13_DEV, OLD_ATLAS_CACHE, CLINICAL_CSV]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Required project artifacts are missing:\n' + '\n'.join(missing))

assert INCLUDE_HISTORICAL_TEST is False
assert set(STAGE11_RECIPES) == set(range(5))
assert sum(EXPECTED_FOLD_COUNTS.values()) == EXPECTED_CASES
assert all(v['connectivity'] == 26 and v['morphology'] == 'none' and v['minvox'] == 10 for v in STAGE11_RECIPES.values())
assert STAGE11_RECIPES[2]['threshold'] == 0.45
assert all(STAGE11_RECIPES[f]['threshold'] == 0.40 for f in [0,1,3,4])

print('✅ Colab CPU/high-RAM Stage12 environment ready')
print('✅ No training code and no neural-inference code are used by this notebook')
print('✅ Output namespace:', OUT_ROOT)


In [ ]:
#@title 1. Helpers and fail-closed frozen-Hybrid namespace resolver { display-mode: "form" }
def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True))
    os.replace(tmp, path)

def scalar_text(x):
    a = np.asarray(x)
    if a.size != 1:
        raise RuntimeError(f'Expected scalar text metadata, got shape={a.shape}')
    v = a.reshape(-1)[0]
    if isinstance(v, bytes):
        v = v.decode('utf-8')
    return str(v).strip()

def fold_from_prediction_path(p, root):
    rel_parts = Path(p).relative_to(root).parts
    hits = []
    for part in rel_parts:
        m = re.fullmatch(r'outer_([0-4])', part)
        if m:
            hits.append(int(m.group(1)))
    if len(hits) != 1:
        raise RuntimeError(f'Cannot resolve exactly one outer fold from {p}')
    return hits[0]

def case_from_outer_pred_path(p):
    suffix = '.outer_pred.npz'
    if not p.name.endswith(suffix):
        raise RuntimeError(f'Unexpected prediction filename: {p.name}')
    case = p.name[:-len(suffix)]
    if not re.fullmatch(r'MSLesSeg_P\d+(?:_T\d+)?', case):
        raise RuntimeError(f'Unexpected case id from prediction filename: {case}')
    return case

def validate_hybrid_root(root):
    root = Path(root)
    if not root.is_dir():
        return False, f'not a directory: {root}', None
    pred_files = sorted(root.glob('outer_folds/outer_*/outer_predictions/*.outer_pred.npz'))
    if len(pred_files) != EXPECTED_CASES:
        return False, f'expected {EXPECTED_CASES} *.outer_pred.npz files, found {len(pred_files)}', None
    records = []
    seen = set()
    fold_counts = {f:0 for f in range(5)}
    try:
        for p in pred_files:
            if 'outer_predictions' not in p.parts:
                raise RuntimeError(f'Prediction is not under an outer_predictions directory: {p}')
            fold = fold_from_prediction_path(p, root)
            case = case_from_outer_pred_path(p)
            if case in seen:
                raise RuntimeError(f'Duplicate case across outer folds: {case}')
            seen.add(case)
            with np.load(p, allow_pickle=False) as z:
                required = {'hybrid','protocol_sha','implementation_id','spacing_xyz'}
                miss = required.difference(z.files)
                if miss:
                    raise RuntimeError(f'{p.name}: missing NPZ fields {sorted(miss)}')
                if scalar_text(z['protocol_sha']) != PROTOCOL_SHA:
                    raise RuntimeError(f'{p.name}: protocol_sha mismatch')
                if scalar_text(z['implementation_id']) != FUSION_IMPLEMENTATION_ID:
                    raise RuntimeError(f'{p.name}: implementation_id mismatch')
                spacing = np.asarray(z['spacing_xyz'], dtype=float).reshape(-1)
                if spacing.size != 3 or not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):
                    raise RuntimeError(f'{p.name}: invalid spacing_xyz={spacing}')
            fold_counts[fold] += 1
            records.append({'case':case, 'fold':fold, 'prediction_path':str(p)})
        if fold_counts != EXPECTED_FOLD_COUNTS:
            raise RuntimeError(f'Fold-count mismatch: {fold_counts} != {EXPECTED_FOLD_COUNTS}')
        if len(seen) != EXPECTED_CASES:
            raise RuntimeError(f'Unique-case mismatch: {len(seen)}')
    except Exception as e:
        return False, repr(e), None
    return True, 'complete provenance-locked Hybrid namespace', records

preferred = ROOT / 'graphms_resencm250_true_hybrid_v3_5_1_8944f1a0de'
candidates = []
if preferred.exists():
    candidates.append(preferred)
for p in sorted(ROOT.glob('graphms_resencm250_true_hybrid_v3_5_1_*')):
    if p not in candidates:
        candidates.append(p)

if not candidates:
    raise RuntimeError('No GraphMS v3.5.1 Hybrid namespace found under ' + str(ROOT))

validation = []
complete = []
for root in candidates:
    ok, reason, records = validate_hybrid_root(root)
    validation.append({'root':str(root), 'accepted':bool(ok), 'reason':reason})
    if ok:
        complete.append((root, records))

display(pd.DataFrame(validation))
if len(complete) != 1:
    raise RuntimeError(f'FAIL CLOSED: expected exactly one complete matching Hybrid namespace, found {len(complete)}')

HYBRID_ROOT, HYBRID_RECORDS = complete[0]
HYBRID_RECORDS = sorted(HYBRID_RECORDS, key=lambda r: (r['fold'], r['case']))
print('✅ Unique frozen Hybrid namespace:', HYBRID_ROOT)
print('✅ 5 folds / 93 unique cases / exact protocol + implementation provenance')


In [ ]:
#@title 2. Reconstruct frozen cross-fitted Stage11-v1 masks and reproduce DSC { display-mode: "form" }
STRUCT26_STAGE11 = ndimage.generate_binary_structure(3, 3)

def stage11_remove_small(mask, minvox):
    lab, n = ndimage.label(np.asarray(mask, dtype=bool), structure=STRUCT26_STAGE11)
    if n == 0:
        return np.zeros_like(mask, dtype=bool)
    sizes = np.bincount(lab.ravel())
    keep = np.where(sizes >= int(minvox))[0]
    keep = keep[keep != 0]
    if len(keep) == 0:
        return np.zeros_like(mask, dtype=bool)
    return np.isin(lab, keep)

def dice_binary(pred, gt):
    pred = np.asarray(pred, dtype=bool)
    gt = np.asarray(gt, dtype=bool)
    den = int(pred.sum() + gt.sum())
    return 1.0 if den == 0 else float(2.0 * np.logical_and(pred, gt).sum() / den)

def check_same_raw_geometry(a, b, label_a, label_b):
    if a.shape != b.shape:
        raise RuntimeError(f'Geometry shape mismatch {label_a}{a.shape} vs {label_b}{b.shape}')
    if not np.allclose(a.affine, b.affine, atol=1e-4, rtol=0):
        raise RuntimeError(f'Geometry affine mismatch: {label_a} vs {label_b}')

expected_mask_names = {f"{r['case']}.nii.gz" for r in HYBRID_RECORDS}
existing_mask_names = {p.name for p in FINAL_MASK_DIR.glob('*.nii.gz')}
extra = sorted(existing_mask_names - expected_mask_names)
if extra:
    raise RuntimeError('FAIL CLOSED: stale/unexpected NIfTI masks exist in final_hybrid_masks: ' + ', '.join(extra[:10]))

ledger_rows = []
gt_reads_for_verification = 0
for rec in HYBRID_RECORDS:
    case = rec['case']
    fold = int(rec['fold'])
    recipe = STAGE11_RECIPES[fold]
    pred_path = Path(rec['prediction_path'])
    flair_path = IMAGES_TR / f'{case}_0000.nii.gz'
    gt_path = LABELS_TR / f'{case}.nii.gz'
    t1_path = IMAGES_TR / f'{case}_0001.nii.gz'
    t2_path = IMAGES_TR / f'{case}_0002.nii.gz'
    for p in [flair_path, t1_path, t2_path, gt_path]:
        if not p.exists():
            raise FileNotFoundError(f'Missing required development artifact: {p}')

    flair_img = nib.load(str(flair_path))
    gt_img = nib.load(str(gt_path))  # GT read occurs only in this verification stage.
    gt_reads_for_verification += 1
    check_same_raw_geometry(flair_img, gt_img, 'FLAIR', 'GT')

    with np.load(pred_path, allow_pickle=False) as z:
        # Provenance was already checked in the namespace resolver; re-check here before consuming the probability map.
        if scalar_text(z['protocol_sha']) != PROTOCOL_SHA or scalar_text(z['implementation_id']) != FUSION_IMPLEMENTATION_ID:
            raise RuntimeError(f'{case}: frozen prediction provenance changed between audit and reconstruction')
        stored_spacing = np.asarray(z['spacing_xyz'], dtype=float).reshape(-1)
        raw_spacing = np.asarray(flair_img.header.get_zooms()[:3], dtype=float)
        if stored_spacing.size != 3 or not np.allclose(stored_spacing, raw_spacing, atol=1e-5, rtol=0):
            raise RuntimeError(f'{case}: spacing_xyz mismatch stored={stored_spacing} raw_FLAIR={raw_spacing}')
        prob = np.asarray(z['hybrid'])

    # Frozen v6 outer probabilities are stored in SimpleITK array order:
    # (z, y, x). nibabel NIfTI arrays use (x, y, z).
    prob_xyz = np.transpose(prob, (2, 1, 0))

    if prob.ndim != 3 or prob_xyz.shape != flair_img.shape:
        raise RuntimeError(
            f'{case}: Hybrid probability geometry mismatch '
            f'v6_zyx={prob.shape}, nib_xyz={prob_xyz.shape}, '
            f'FLAIR={flair_img.shape}'
        )
    if not np.all(np.isfinite(prob_xyz)):
        raise RuntimeError(f'{case}: Hybrid probability map contains non-finite values')
    if float(prob_xyz.min()) < -1e-6 or float(prob_xyz.max()) > 1.0 + 1e-6:
        raise RuntimeError(f'{case}: Hybrid probability values outside [0,1]')

    binary = prob_xyz >= float(recipe['threshold'])
    final_mask = stage11_remove_small(binary, recipe['minvox'])
    out_mask = FINAL_MASK_DIR / f'{case}.nii.gz'

    hdr = flair_img.header.copy()
    hdr.set_data_dtype(np.uint8)
    out_img = nib.Nifti1Image(final_mask.astype(np.uint8), flair_img.affine, header=hdr)
    # Geometry is copied from raw FLAIR, never from GT.
    try:
        out_img.set_qform(flair_img.get_qform(), int(flair_img.header['qform_code']))
        out_img.set_sform(flair_img.get_sform(), int(flair_img.header['sform_code']))
    except Exception:
        pass
    nib.save(out_img, str(out_mask))

    saved_img = nib.load(str(out_mask))
    check_same_raw_geometry(flair_img, saved_img, 'FLAIR', 'saved final mask')
    saved_mask = np.asarray(saved_img.dataobj) >= 0.5
    if not np.array_equal(saved_mask, final_mask):
        raise RuntimeError(f'{case}: saved NIfTI mask differs from reconstructed mask')

    gt = np.asarray(gt_img.dataobj) > 0.5
    dsc = dice_binary(final_mask, gt)
    ledger_rows.append({
        'case':case, 'fold':fold, 'threshold':recipe['threshold'], 'minvox':recipe['minvox'],
        'connectivity':recipe['connectivity'], 'morphology':recipe['morphology'], 'DSC':dsc,
        'mask_path':str(out_mask), 'mask_sha256':sha256_file(out_mask),
        'prediction_path':str(pred_path), 'protocol_sha':PROTOCOL_SHA,
        'implementation_id':FUSION_IMPLEMENTATION_ID,
    })
    del prob, prob_xyz, binary, final_mask, saved_mask, gt, gt_img, flair_img, saved_img

ledger = pd.DataFrame(ledger_rows).sort_values(['fold','case']).reset_index(drop=True)
if len(ledger) != EXPECTED_CASES or ledger['case'].nunique() != EXPECTED_CASES:
    raise RuntimeError('Final mask ledger is not exactly 93 unique cases')
actual_counts = ledger.groupby('fold').size().to_dict()
if actual_counts != EXPECTED_FOLD_COUNTS:
    raise RuntimeError(f'Final mask fold counts changed: {actual_counts}')
fold_means = ledger.groupby('fold')['DSC'].mean().sort_index()
equal_fold_dsc = float(fold_means.mean())
if abs(equal_fold_dsc - EXPECTED_EQUAL_FOLD_DSC) > DSC_TOL:
    raise RuntimeError(
        f'FAIL CLOSED: frozen segmentation DSC did not reproduce. '
        f'Observed={equal_fold_dsc:.12f}, expected={EXPECTED_EQUAL_FOLD_DSC:.12f}, tol={DSC_TOL}'
    )
if gt_reads_for_verification != EXPECTED_CASES:
    raise RuntimeError(f'GT verification read count mismatch: {gt_reads_for_verification}')

ledger_path = OUT_ROOT / 'FINAL_HYBRID_MASK_LEDGER.csv'
ledger.to_csv(ledger_path, index=False)

seg_handoff = {
    'status':'FROZEN',
    'claim_scope':'development five-fold CV',
    'model':'GraphMS v3.5.1 Hybrid',
    'protocol_sha':PROTOCOL_SHA,
    'fusion_implementation_id':FUSION_IMPLEMENTATION_ID,
    'inference':'original uniform-overlap Hybrid inference; stored probabilities only; no new neural inference',
    'stage11':'cross-fitted Stage11-v1',
    'stage11_fold_recipes':{str(k):v for k,v in STAGE11_RECIPES.items()},
    'fold_counts':{str(k):v for k,v in EXPECTED_FOLD_COUNTS.items()},
    'n_cases':EXPECTED_CASES,
    'equal_fold_dsc':equal_fold_dsc,
    'equal_fold_dsc_expected':EXPECTED_EQUAL_FOLD_DSC,
    'dice_tolerance':DSC_TOL,
    'ground_truth_usage':'GT read only for frozen segmentation verification; GT is not a Stage12 feature input',
    'gt_files_read_for_verification':gt_reads_for_verification,
    'mask_geometry_source':'raw FLAIR NIfTI',
    'hybrid_probability_root':str(HYBRID_ROOT),
    'new_training_performed':False,
    'new_neural_inference_performed':False,
}
seg_handoff_path = OUT_ROOT / 'FINAL_SEGMENTATION_HANDOFF.json'
atomic_json(seg_handoff_path, seg_handoff)

# Bind all downstream canonical Stage12 cache provenance to the
# FINAL GraphMS Hybrid segmentation handoff rather than the
# historical ResEncM WINNING_RECIPE.json.
WINNING_RECIPE = seg_handoff_path

print('✅ Frozen Stage11 masks:', len(ledger), '/', EXPECTED_CASES)
print('✅ Fold means:')
display(fold_means.rename('DSC').to_frame())
print(f'✅ Equal-fold DSC reproduced: {equal_fold_dsc:.12f}')
print('✅ GT use ends here. No later cell loads labelsTr or passes GT to extract_case().')


## 3. Canonical Stage12 v2.1 feature engine

The following code cell is the supplied audited `GraphMSNet-Stage12-MAX-v2.1` feature engine. Its actual feature equations and feature families are preserved. The only required source-level provenance edit inside `extract_case()` is:

`GraphMS v3.5.1 Hybrid + cross-fitted Stage11-v1 + 26-connectivity + no morphology`

The canonical safety filter remains `>=0.5` + 26-connectivity + minimum 5 voxels. Because every frozen Stage11 mask is already binary and has minimum component size 10, the next cell explicitly verifies that this safety filter is idempotent before any features are extracted.


In [ ]:
#@title Stage 12 backend definition — canonical v2.1 feature extractor { display-mode: "form" }
# Canonical Stage-12 v2.1 backend configuration preserved for the final Hybrid handoff.
SEG_THRESHOLD=.5; MIN_COMPONENT_VOXELS=5; CONNECTIVITY=26; FEATURE_SCHEMA_VERSION='GraphMSNet-Stage12-MAX-v2.1'; GLCM_Z_CLIP=5.; GLCM_BIN_WIDTH_Z=.25; GLCM_DISTANCES=(1,2); GLCM_MIN_VOXELS=8; ENABLE_ATLAS_LOCATION=True; FORCE_RECOMPUTE=False; CHECKPOINT_VERSION='graphms-v3.5.1-stage12-final-hybrid-v1'; CACHE_DIR=STAGE12_DEMO_CACHE; ATLAS_CACHE=STAGE12/'atlas_cache'; MODALITIES=['flair','t1','t2']; IMAGES_TR=RAW/'imagesTr'; IMAGES_TS=RAW/'imagesTs'
def sha256_small(path,chunk=1<<20):return sha256_file(path,chunk)
def case_from_mask_path(p):return p.name[:-7] if p.name.endswith('.nii.gz') else p.stem
def image_triplet(case_id,role):
    base=IMAGES_TR if role=='development_oof' else IMAGES_TS; return [base/f'{case_id}_{i:04d}.nii.gz' for i in range(3)]


# 3. Core feature engine: filter5, robust intensity, 3D shape, guide shape, sparse 3D GLCM.

STRUCT26 = ndimage.generate_binary_structure(3, 3)
EPS = 1e-12

def canonical_img(path):
    return nib.as_closest_canonical(nib.load(str(path)))

def filter5(mask):
    lab, n = ndimage.label(mask.astype(bool), structure=STRUCT26)
    if n == 0:
        return np.zeros_like(mask, dtype=bool)
    sizes = np.bincount(lab.ravel())
    keep = np.where(sizes >= MIN_COMPONENT_VOXELS)[0]
    keep = keep[keep != 0]
    if len(keep) == 0:
        return np.zeros_like(mask, dtype=bool)
    return np.isin(lab, keep)

def robust_brain_scale(arr):
    a = np.asarray(arr, dtype=np.float32)
    brain = np.isfinite(a) & (np.abs(a) > EPS)
    vals = a[brain]
    if vals.size < 32:
        vals = a[np.isfinite(a)]
    if vals.size == 0:
        return 0.0, 1.0, brain
    med = float(np.median(vals))
    mad = float(np.median(np.abs(vals - med))) * 1.4826
    if mad < 1e-6:
        q25, q75 = np.percentile(vals, [25, 75])
        mad = float((q75 - q25) / 1.349)
    if mad < 1e-6:
        mad = float(np.std(vals))
    if mad < 1e-6:
        mad = 1.0
    return med, mad, brain

def first_order_features(values, prefix):
    v = np.asarray(values, dtype=np.float64)
    v = v[np.isfinite(v)]
    keys = ["mean", "std", "median", "p10", "p25", "p75", "p90", "iqr", "skew", "kurtosis"]
    if v.size == 0:
        return {f"{prefix}_{k}": np.nan for k in keys}
    q10, q25, q50, q75, q90 = np.percentile(v, [10, 25, 50, 75, 90])
    return {
        f"{prefix}_mean": float(np.mean(v)),
        f"{prefix}_std": float(np.std(v)),
        f"{prefix}_median": float(q50),
        f"{prefix}_p10": float(q10),
        f"{prefix}_p25": float(q25),
        f"{prefix}_p75": float(q75),
        f"{prefix}_p90": float(q90),
        f"{prefix}_iqr": float(q75 - q25),
        f"{prefix}_skew": float(stats.skew(v, bias=False)) if v.size >= 3 else np.nan,
        f"{prefix}_kurtosis": float(stats.kurtosis(v, fisher=True, bias=False)) if v.size >= 4 else np.nan,
    }

def glcm_directions(distance=1):
    # 13 unique half-space directions; symmetric accumulation represents all 26 neighbours.
    dirs = []
    for dx in range(-distance, distance + 1, distance):
        for dy in range(-distance, distance + 1, distance):
            for dz in range(-distance, distance + 1, distance):
                if dx == dy == dz == 0:
                    continue
                g = math.gcd(math.gcd(abs(dx), abs(dy)), abs(dz))
                if g != distance:
                    continue
                if (dx > 0) or (dx == 0 and dy > 0) or (dx == 0 and dy == 0 and dz > 0):
                    dirs.append((dx, dy, dz))
    return dirs


def _overlap_slices(shape, delta):
    src, dst = [], []
    for n, d in zip(shape, delta):
        if d >= 0:
            src.append(slice(0, n - d))
            dst.append(slice(d, n))
        else:
            src.append(slice(-d, n))
            dst.append(slice(0, n + d))
    return tuple(src), tuple(dst)

def sparse_glcm_features(zarr, mask, distance=1, prefix="flair_glcm_d1"):
    """Symmetric 3D lesion-ROI GLCM, vectorized for low wall-clock time."""
    names = ["contrast", "dissimilarity", "homogeneity", "asm", "energy",
             "correlation", "entropy", "pair_count"]
    if int(mask.sum()) < GLCM_MIN_VOXELS:
        return {f"{prefix}_{n}": (0.0 if n == "pair_count" else np.nan) for n in names}

    z = np.nan_to_num(
        np.clip(np.asarray(zarr, dtype=np.float32), -GLCM_Z_CLIP, GLCM_Z_CLIP),
        nan=0.0, posinf=GLCM_Z_CLIP, neginf=-GLCM_Z_CLIP
    )
    n_bins = int(round((2 * GLCM_Z_CLIP) / GLCM_BIN_WIDTH_Z)) + 1
    q = np.floor((z + GLCM_Z_CLIP) / GLCM_BIN_WIDTH_Z).astype(np.int16)
    q = np.clip(q, 0, n_bins - 1)

    counts = np.zeros(n_bins * n_bins, dtype=np.int64)
    pair_count = 0

    for delta in glcm_directions(distance):
        s0, s1 = _overlap_slices(mask.shape, delta)
        valid = mask[s0] & mask[s1]
        if not np.any(valid):
            continue
        a = q[s0][valid].astype(np.int64)
        b = q[s1][valid].astype(np.int64)
        counts += np.bincount(a * n_bins + b, minlength=n_bins*n_bins)
        counts += np.bincount(b * n_bins + a, minlength=n_bins*n_bins)
        pair_count += 2 * len(a)

    if pair_count == 0:
        return {f"{prefix}_{n}": (0.0 if n == "pair_count" else np.nan) for n in names}

    P = counts.reshape(n_bins, n_bins).astype(np.float64)
    P /= P.sum()
    I, J = np.indices(P.shape)
    diff = I - J

    contrast = float(np.sum(P * diff**2))
    dissimilarity = float(np.sum(P * np.abs(diff)))
    homogeneity = float(np.sum(P / (1.0 + diff**2)))
    asm = float(np.sum(P**2))
    energy = float(np.sqrt(asm))
    nz = P[P > 0]
    entropy = float(-np.sum(nz * np.log2(nz)))

    pi, pj = P.sum(axis=1), P.sum(axis=0)
    axis = np.arange(n_bins, dtype=float)
    mi, mj = float(np.sum(axis*pi)), float(np.sum(axis*pj))
    si = float(np.sqrt(np.sum(((axis-mi)**2)*pi)))
    sj = float(np.sqrt(np.sum(((axis-mj)**2)*pj)))
    corr = float(np.sum(P*(I-mi)*(J-mj))/(si*sj+EPS)) if si > 0 and sj > 0 else np.nan

    return {
        f"{prefix}_contrast": contrast,
        f"{prefix}_dissimilarity": dissimilarity,
        f"{prefix}_homogeneity": homogeneity,
        f"{prefix}_asm": asm,
        f"{prefix}_energy": energy,
        f"{prefix}_correlation": corr,
        f"{prefix}_entropy": entropy,
        f"{prefix}_pair_count": float(pair_count),
    }

def surface_area_mm2(mask, spacing):
    if int(mask.sum()) < 4:
        return np.nan
    idx = np.argwhere(mask)
    lo = np.maximum(idx.min(axis=0) - 1, 0)
    hi = np.minimum(idx.max(axis=0) + 2, np.array(mask.shape))
    crop = mask[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]].astype(np.uint8)
    if min(crop.shape) < 2:
        return np.nan
    try:
        verts, faces, _, _ = measure.marching_cubes(crop, level=0.5, spacing=spacing)
        return float(measure.mesh_surface_area(verts, faces))
    except Exception:
        return np.nan

def guide_2d_shape(mask, spacing):
    if not mask.any():
        return {
            "guide_max_slice_area_mm2": 0.0,
            "guide_max_slice_perimeter_mm": 0.0,
            "guide_max_slice_compactness": np.nan,
        }
    sx, sy, _ = spacing
    areas = mask.sum(axis=(0, 1))
    z = int(np.argmax(areas))
    sl = mask[:, :, z]
    area = float(sl.sum() * sx * sy)
    per_px = float(measure.perimeter_crofton(sl.astype(np.uint8), directions=4))
    perimeter = per_px * float((sx + sy) / 2.0)
    compact = float(4.0 * math.pi * area / (perimeter**2 + EPS)) if perimeter > 0 else np.nan
    return {
        "guide_max_slice_area_mm2": area,
        "guide_max_slice_perimeter_mm": perimeter,
        "guide_max_slice_compactness": compact,
    }

def pca_shape(mask, spacing):
    coords = np.argwhere(mask)
    if len(coords) < 4:
        return {"shape_elongation": np.nan, "shape_flatness": np.nan}
    pts = coords.astype(np.float64) * np.asarray(spacing)[None, :]
    if len(pts) > 100000:
        take = np.linspace(0, len(pts) - 1, 100000).astype(int)
        pts = pts[take]
    ev = np.linalg.eigvalsh(np.cov(pts.T))
    ev = np.sort(np.maximum(ev, 0))[::-1]
    if ev[0] <= EPS:
        return {"shape_elongation": np.nan, "shape_flatness": np.nan}
    return {
        "shape_elongation": float(np.sqrt(ev[1] / ev[0])) if len(ev) > 1 else np.nan,
        "shape_flatness": float(np.sqrt(ev[2] / ev[0])) if len(ev) > 2 else np.nan,
    }

print("Core feature engine ready.")


# 4. Optional atlas / periventricular context.
# Highest-value extension, but fail-open: core Stage 12 must still finish if atlas download/API fails.
ATLAS = {"enabled": False, "reason": "disabled"}

def ensure_nilearn():
    import importlib.util, subprocess, sys
    if importlib.util.find_spec("nilearn") is None:
        print("Installing nilearn for atlas features...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nilearn"])

def as_img(x):
    from nilearn.image import load_img
    return load_img(x)

def normalise_label_name(x):
    return re.sub(r"[^a-z0-9]+", " ", str(x).lower()).strip()

if ENABLE_ATLAS_LOCATION:
    try:
        ensure_nilearn()
        from nilearn.datasets import fetch_atlas_harvard_oxford

        cort = fetch_atlas_harvard_oxford(
            "cort-maxprob-thr25-2mm", data_dir=str(ATLAS_CACHE), symmetric_split=False
        )
        sub = fetch_atlas_harvard_oxford(
            "sub-maxprob-thr25-2mm", data_dir=str(ATLAS_CACHE), symmetric_split=False
        )

        cort_img = as_img(cort.maps)
        sub_img = as_img(sub.maps)
        cort_arr = np.asarray(cort_img.dataobj)
        sub_arr = np.asarray(sub_img.dataobj)
        cort_aff = np.asarray(cort_img.affine)
        sub_aff = np.asarray(sub_img.affine)

        sub_labels = [normalise_label_name(x) for x in sub.labels]
        vent_ids = [
            i for i, name in enumerate(sub_labels)
            if ("lateral ventricle" in name) or ("inferior lateral ventricle" in name)
        ]
        brainstem_ids = [i for i, name in enumerate(sub_labels) if ("brain stem" in name or "brainstem" in name)]

        vent_mask = np.isin(sub_arr, vent_ids) if vent_ids else np.zeros_like(sub_arr, dtype=bool)
        sub_zooms = sub_img.header.get_zooms()[:3]
        vent_dist = ndimage.distance_transform_edt(~vent_mask, sampling=sub_zooms) if vent_mask.any() else None

        ATLAS = {
            "enabled": True,
            "cort_arr": cort_arr,
            "sub_arr": sub_arr,
            "cort_aff": cort_aff,
            "sub_aff": sub_aff,
            "cort_labels": list(cort.labels),
            "sub_labels": list(sub.labels),
            "brainstem_ids": brainstem_ids,
            "vent_dist": vent_dist,
            "source": "Harvard-Oxford maxprob-thr25-2mm",
        }
        print("Atlas features enabled:", ATLAS["source"])
    except Exception as e:
        ATLAS = {"enabled": False, "reason": repr(e)}
        print("WARNING: atlas features disabled; core Stage 12 will continue.")
        print("Reason:", repr(e))
else:
    print("Atlas features disabled by configuration.")

def sample_world_array(world_xyz, arr, affine):
    if len(world_xyz) == 0:
        return np.array([], dtype=arr.dtype), np.array([], dtype=bool)
    inv = np.linalg.inv(affine)
    hom = np.c_[world_xyz, np.ones(len(world_xyz))]
    ijk = np.rint((inv @ hom.T).T[:, :3]).astype(int)
    valid = np.all((ijk >= 0) & (ijk < np.asarray(arr.shape)[None, :]), axis=1)
    vals = np.zeros(len(ijk), dtype=arr.dtype)
    v = ijk[valid]
    if len(v):
        vals[valid] = arr[v[:, 0], v[:, 1], v[:, 2]]
    return vals, valid

def atlas_features(mask, mask_affine, voxel_volume_mm3):
    base = {
        "atlas_inbounds_fraction": np.nan,
        "atlas_cortical_overlap_mm3": np.nan,
        "atlas_subcortical_overlap_mm3": np.nan,
        "atlas_brainstem_overlap_mm3": np.nan,
        "periventricular_fraction_le3mm": np.nan,
        "periventricular_fraction_le5mm": np.nan,
        "periventricular_fraction_le10mm": np.nan,
        "ventricle_distance_mean_mm": np.nan,
        "ventricle_distance_min_mm": np.nan,
    }
    if not ATLAS.get("enabled") or not mask.any():
        return base

    vox = np.argwhere(mask)
    hom = np.c_[vox, np.ones(len(vox))]
    world = (mask_affine @ hom.T).T[:, :3]

    cort_vals, cort_valid = sample_world_array(world, ATLAS["cort_arr"], ATLAS["cort_aff"])
    sub_vals, sub_valid = sample_world_array(world, ATLAS["sub_arr"], ATLAS["sub_aff"])
    valid = cort_valid & sub_valid
    inb = float(valid.mean()) if len(valid) else 0.0
    base["atlas_inbounds_fraction"] = inb

    # Guard against silently sampling a clearly incompatible coordinate system.
    if inb < 0.80:
        return base

    c = cort_vals[valid]
    s = sub_vals[valid]
    base["atlas_cortical_overlap_mm3"] = float(np.sum(c > 0) * voxel_volume_mm3)
    base["atlas_subcortical_overlap_mm3"] = float(np.sum(s > 0) * voxel_volume_mm3)
    if ATLAS["brainstem_ids"]:
        base["atlas_brainstem_overlap_mm3"] = float(
            np.sum(np.isin(s, ATLAS["brainstem_ids"])) * voxel_volume_mm3
        )

    if ATLAS.get("vent_dist") is not None:
        dvals, dvalid = sample_world_array(world, ATLAS["vent_dist"], ATLAS["sub_aff"])
        d = dvals[dvalid].astype(float)
        if len(d):
            base["periventricular_fraction_le3mm"] = float(np.mean(d <= 3.0))
            base["periventricular_fraction_le5mm"] = float(np.mean(d <= 5.0))
            base["periventricular_fraction_le10mm"] = float(np.mean(d <= 10.0))
            base["ventricle_distance_mean_mm"] = float(np.mean(d))
            base["ventricle_distance_min_mm"] = float(np.min(d))
    return base


# 5. Case-level + lesion-level extractor.
MODALITIES = ["flair", "t1", "t2"]

def assert_same_geometry(a, b, name_a="mask", name_b="image"):
    if a.shape != b.shape:
        raise ValueError(f"Geometry mismatch {name_a}{a.shape} vs {name_b}{b.shape}")
    if not np.allclose(a.affine, b.affine, atol=1e-3, rtol=0):
        raise ValueError(f"Affine mismatch between {name_a} and {name_b}")

def parse_case_id(case_id):
    m = re.match(r"^MSLesSeg_(P\d+)(?:_(T\d+))?$", case_id)
    if not m:
        raise ValueError(f"Unexpected case id: {case_id}")
    patient = m.group(1)
    timepoint = m.group(2) or "T1"
    return patient, timepoint

def lesion_component_table(case_id, role, mask, affine, spacing):
    lab, n = ndimage.label(mask, structure=STRUCT26)
    voxel_vol = float(np.prod(spacing))
    rows = []
    for k in range(1, n + 1):
        idx = np.argwhere(lab == k)
        if len(idx) == 0:
            continue
        hom = np.c_[idx, np.ones(len(idx))]
        world = (affine @ hom.T).T[:, :3]
        lo = idx.min(axis=0); hi = idx.max(axis=0) + 1
        rows.append({
            "case_id": case_id,
            "role": role,
            "lesion_id": int(k),
            "lesion_voxels": int(len(idx)),
            "lesion_volume_mm3": float(len(idx) * voxel_vol),
            "centroid_world_x_mm": float(world[:, 0].mean()),
            "centroid_world_y_mm": float(world[:, 1].mean()),
            "centroid_world_z_mm": float(world[:, 2].mean()),
            "bbox_x_mm": float((hi[0] - lo[0]) * spacing[0]),
            "bbox_y_mm": float((hi[1] - lo[1]) * spacing[1]),
            "bbox_z_mm": float((hi[2] - lo[2]) * spacing[2]),
        })
    return rows


def atomic_write_text(path, text):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text)
    os.replace(tmp, path)

def source_stat_signature(path):
    st = Path(path).stat()
    return {"name": Path(path).name, "size": int(st.st_size), "mtime_ns": int(st.st_mtime_ns)}

def feature_config_fingerprint():
    payload = {
        "feature_schema_version": FEATURE_SCHEMA_VERSION,
        "checkpoint_version": CHECKPOINT_VERSION,
        "seg_threshold": SEG_THRESHOLD,
        "min_component_voxels": MIN_COMPONENT_VOXELS,
        "connectivity": CONNECTIVITY,
        "glcm_z_clip": GLCM_Z_CLIP,
        "glcm_bin_width_z": GLCM_BIN_WIDTH_Z,
        "glcm_distances": list(GLCM_DISTANCES),
        "glcm_min_voxels": GLCM_MIN_VOXELS,
        "atlas_enabled_runtime": bool(ATLAS.get("enabled")),
        "atlas_source": ATLAS.get("source"),
        "winner_recipe_sha256": sha256_small(WINNING_RECIPE),
    }
    return hashlib.sha256(json.dumps(payload,sort_keys=True,separators=(",",":")).encode()).hexdigest()

CONFIG_FINGERPRINT = feature_config_fingerprint()

def case_input_fingerprint(mask_path, role):
    case_id = case_from_mask_path(mask_path)
    payload = {
        "case_id": case_id,
        "role": role,
        "config": CONFIG_FINGERPRINT,
        "mask_sha256": sha256_small(mask_path),
        "mri_sources": [source_stat_signature(p) for p in image_triplet(case_id, role)],
    }
    return hashlib.sha256(json.dumps(payload,sort_keys=True,separators=(",",":")).encode()).hexdigest()

def case_cache_paths(case_id):
    return CACHE_DIR / f"{case_id}.json", CACHE_DIR / f"{case_id}_lesions.json"

def extract_case(mask_path, role):
    case_id = case_from_mask_path(mask_path)
    cpath, lpath = case_cache_paths(case_id)
    input_fp = case_input_fingerprint(mask_path, role)
    if cpath.exists() and lpath.exists() and not FORCE_RECOMPUTE:
        rec = json.loads(cpath.read_text())
        if (
            rec.get("_feature_schema_version") == FEATURE_SCHEMA_VERSION
            and rec.get("_input_fingerprint") == input_fp
            and rec.get("_config_fingerprint") == CONFIG_FINGERPRINT
        ):
            return rec, json.loads(lpath.read_text())

    t0 = time.perf_counter()
    patient, timepoint = parse_case_id(case_id)

    mask_img = canonical_img(mask_path)
    mask_prob_or_label = np.asarray(mask_img.dataobj)
    # CV/test folders contain labels; thresholding is harmless and keeps the contract explicit.
    mask = filter5(mask_prob_or_label >= SEG_THRESHOLD)
    spacing = tuple(float(x) for x in mask_img.header.get_zooms()[:3])
    voxel_vol = float(np.prod(spacing))
    lesion_voxels = int(mask.sum())
    lesion_volume_mm3 = float(lesion_voxels * voxel_vol)

    lab, n_comp = ndimage.label(mask, structure=STRUCT26)
    sizes_vox = np.bincount(lab.ravel())[1:] if n_comp else np.array([], dtype=int)
    sizes_mm3 = sizes_vox.astype(float) * voxel_vol
    sort_sizes = np.sort(sizes_mm3)[::-1] if len(sizes_mm3) else np.array([], dtype=float)

    rec = {
        "_feature_schema_version": FEATURE_SCHEMA_VERSION,
        "_input_fingerprint": input_fp,
        "_config_fingerprint": CONFIG_FINGERPRINT,
        "case_id": case_id,
        "patient_id": patient,
        "timepoint": timepoint,
        "role": role,
        "mask_source": str(mask_path),
        "frozen_segmentation": "GraphMS v3.5.1 Hybrid + cross-fitted Stage11-v1 + 26-connectivity + no morphology",
        "ground_truth_mask_used": False,
        "lesion_voxels": lesion_voxels,
        "lesion_volume_mm3": lesion_volume_mm3,
        "lesion_volume_ml": lesion_volume_mm3 / 1000.0,
        "log_lesion_volume": float(np.log1p(lesion_volume_mm3)),
        "lesion_count": int(n_comp),
        "largest_lesion_volume_mm3": float(sort_sizes[0]) if len(sort_sizes) else 0.0,
        "second_largest_lesion_volume_mm3": float(sort_sizes[1]) if len(sort_sizes) > 1 else 0.0,
        "mean_lesion_volume_mm3": float(np.mean(sizes_mm3)) if len(sizes_mm3) else 0.0,
        "median_lesion_volume_mm3": float(np.median(sizes_mm3)) if len(sizes_mm3) else 0.0,
        "std_lesion_volume_mm3": float(np.std(sizes_mm3)) if len(sizes_mm3) else 0.0,
        "largest_lesion_fraction": float(sort_sizes[0] / lesion_volume_mm3) if lesion_volume_mm3 > 0 else 0.0,
        "lesion_count_lt50mm3": int(np.sum(sizes_mm3 < 50.0)),
        "lesion_count_50_200mm3": int(np.sum((sizes_mm3 >= 50.0) & (sizes_mm3 < 200.0))),
        "lesion_count_ge200mm3": int(np.sum(sizes_mm3 >= 200.0)),
        "voxel_volume_mm3": voxel_vol,
        "spacing_x_mm": spacing[0],
        "spacing_y_mm": spacing[1],
        "spacing_z_mm": spacing[2],
    }

    # Global 3D morphology.
    area = surface_area_mm2(mask, spacing)
    rec["shape_surface_area_mm2"] = area
    rec["shape_sphericity"] = (
        float((math.pi ** (1/3)) * ((6.0 * lesion_volume_mm3) ** (2/3)) / (area + EPS))
        if lesion_volume_mm3 > 0 and np.isfinite(area) and area > 0 else np.nan
    )
    rec["shape_compactness_3d"] = (
        float(36.0 * math.pi * lesion_volume_mm3**2 / (area**3 + EPS))
        if lesion_volume_mm3 > 0 and np.isfinite(area) and area > 0 else np.nan
    )
    rec.update(guide_2d_shape(mask, spacing))
    rec.update(pca_shape(mask, spacing))

    # Spatial burden in world coordinates. These are cheap and useful even without an atlas.
    vox = np.argwhere(mask)
    if len(vox):
        hom = np.c_[vox, np.ones(len(vox))]
        world = (mask_img.affine @ hom.T).T[:, :3]
        rec.update({
            "lesion_centroid_world_x_mm": float(world[:, 0].mean()),
            "lesion_centroid_world_y_mm": float(world[:, 1].mean()),
            "lesion_centroid_world_z_mm": float(world[:, 2].mean()),
            "lesion_fraction_left_xlt0": float(np.mean(world[:, 0] < 0)),
            "lesion_fraction_right_xge0": float(np.mean(world[:, 0] >= 0)),
            "lesion_world_x_std_mm": float(world[:, 0].std()),
            "lesion_world_y_std_mm": float(world[:, 1].std()),
            "lesion_world_z_std_mm": float(world[:, 2].std()),
        })
    else:
        for k in [
            "lesion_centroid_world_x_mm","lesion_centroid_world_y_mm","lesion_centroid_world_z_mm",
            "lesion_fraction_left_xlt0","lesion_fraction_right_xge0",
            "lesion_world_x_std_mm","lesion_world_y_std_mm","lesion_world_z_std_mm"
        ]:
            rec[k] = np.nan

    rec.update(atlas_features(mask, mask_img.affine, voxel_vol))

    # MRI features. Load one modality at a time to limit RAM.
    triplet = image_triplet(case_id, role)
    approx_brain_volume_mm3 = np.nan

    for mod, img_path in zip(MODALITIES, triplet):
        img = canonical_img(img_path)
        assert_same_geometry(mask_img, img, "mask", mod)
        arr = np.asarray(img.dataobj, dtype=np.float32)
        med, scale, brain = robust_brain_scale(arr)
        zarr = (arr - med) / scale

        lesion_raw = arr[mask]
        lesion_z = zarr[mask]
        rec.update(first_order_features(lesion_raw, f"{mod}_raw"))
        rec.update(first_order_features(lesion_z, f"{mod}_z"))

        rec[f"{mod}_brain_center"] = med
        rec[f"{mod}_brain_robust_scale"] = scale

        for d in GLCM_DISTANCES:
            rec.update(
                sparse_glcm_features(
                    zarr, mask, distance=d, prefix=f"{mod}_glcm_d{d}"
                )
            )

        if mod == "flair":
            approx_brain_volume_mm3 = float(brain.sum() * voxel_vol)

        del arr, zarr

    rec["approx_brain_volume_mm3"] = approx_brain_volume_mm3
    rec["lesion_volume_normalized"] = (
        float(lesion_volume_mm3 / approx_brain_volume_mm3)
        if np.isfinite(approx_brain_volume_mm3) and approx_brain_volume_mm3 > 0 else np.nan
    )

    lesions = lesion_component_table(case_id, role, mask, mask_img.affine, spacing)
    rec["runtime_sec"] = float(time.perf_counter() - t0)

    # JSON-safe cache.
    clean = {}
    for k, v in rec.items():
        if isinstance(v, (np.floating, float)):
            clean[k] = None if not np.isfinite(v) else float(v)
        elif isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (bool, np.bool_)):
            clean[k] = bool(v)
        else:
            clean[k] = v
    atomic_write_text(cpath, json.dumps(clean, indent=2))
    atomic_write_text(lpath, json.dumps(lesions, indent=2))
    return clean, lesions

print("Case extractor ready.")


In [ ]:
#@title 4. Canonical-engine integrity gates: filter idempotence, atlas, schema contract { display-mode: "form" }
if FEATURE_SCHEMA_VERSION != FEATURE_SCHEMA_VERSION_LOCK:
    raise RuntimeError(f'Feature schema mismatch: {FEATURE_SCHEMA_VERSION}')
if (SEG_THRESHOLD, MIN_COMPONENT_VOXELS, CONNECTIVITY) != (0.5, 5, 26):
    raise RuntimeError('Canonical Stage12 safety-filter configuration changed unexpectedly')

# 4a. Confirm canonical internal safety filter is mathematically idempotent on every final Stage11 mask.
idempotence_failures = []
for p in sorted(FINAL_MASK_DIR.glob('*.nii.gz')):
    img = canonical_img(p)
    raw = np.asarray(img.dataobj)
    before = raw >= 0.5
    after = filter5(before)
    if not np.array_equal(before, after):
        idempotence_failures.append(p.name)
if idempotence_failures:
    raise RuntimeError('FAIL CLOSED: canonical filter5 changed final Stage11 masks: ' + ', '.join(idempotence_failures[:10]))

# 4b. Load the established Stage13 feature-column contract from the old canonical package.
contract_ref = json.loads(OLD_STAGE12_CONTRACT.read_text())
if contract_ref.get('feature_schema_version') != FEATURE_SCHEMA_VERSION_LOCK:
    raise RuntimeError('Old Stage13 feature contract schema version mismatch')
required_imaging_features = list(contract_ref.get('recommended_imaging_features', []))
required_clinical_features = list(contract_ref.get('recommended_clinical_features', []))
required_targets = list(contract_ref.get('targets', []))
if not required_imaging_features or required_targets != ['target_edss','target_edss_ge4']:
    raise RuntimeError('Established Stage13 contract is missing/changed')

# The established contract explicitly depends on atlas/periventricular features; therefore atlas failure is fatal here.
atlas_contract_required = any(
    c.startswith(('atlas_','periventricular_','ventricle_')) for c in required_imaging_features
)
if atlas_contract_required and not ATLAS.get('enabled'):
    raise RuntimeError('FAIL CLOSED: canonical Harvard-Oxford atlas/periventricular context could not be loaded: ' + str(ATLAS.get('reason')))

print('✅ Canonical Stage12 safety filter is idempotent for all 93 final masks')
print('✅ Harvard-Oxford atlas/periventricular context:', 'ACTIVE' if ATLAS.get('enabled') else 'not required')
print('✅ Established Stage13 imaging contract columns:', len(required_imaging_features))


In [ ]:
#@title 5. Run canonical Stage12 on all 93 FINAL Hybrid masks — resumable CPU extraction { display-mode: "form" }
mask_by_case = {p.name[:-7] if p.name.endswith('.nii.gz') else p.stem: p for p in FINAL_MASK_DIR.glob('*.nii.gz')}
expected_cases = set(ledger['case'])
if set(mask_by_case) != expected_cases:
    raise RuntimeError('Final mask directory does not contain exactly the 93 ledger cases')

case_records = []
lesion_records = []
for i, case in enumerate(sorted(expected_cases), start=1):
    rec, lesions = extract_case(mask_by_case[case], 'development_oof')
    case_records.append(rec)
    lesion_records.extend(lesions)
    if i == 1 or i % 10 == 0 or i == EXPECTED_CASES:
        print(f'  Stage12 cases complete: {i}/{EXPECTED_CASES}')

features = pd.DataFrame(case_records).sort_values('case_id').reset_index(drop=True)
lesions = pd.DataFrame(lesion_records)

if len(features) != EXPECTED_CASES or features['case_id'].nunique() != EXPECTED_CASES:
    raise RuntimeError(f'Stage12 feature row gate failed: rows={len(features)}, unique={features.case_id.nunique()}')
if features['patient_id'].nunique() != EXPECTED_PATIENTS:
    raise RuntimeError(f'Stage12 patient gate failed: {features.patient_id.nunique()} != {EXPECTED_PATIENTS}')
if set(features['role'].unique()) != {'development_oof'}:
    raise RuntimeError('Historical/non-development role leaked into refreshed Hybrid Stage12 table')
if not features['ground_truth_mask_used'].eq(False).all():
    raise RuntimeError('GT-use gate failed: one or more Stage12 records claim ground_truth_mask_used != False')
if not features['mask_source'].map(lambda s: str(FINAL_MASK_DIR) in str(s)).all():
    raise RuntimeError('One or more Stage12 rows did not use the refreshed final Hybrid mask directory')
if not features['frozen_segmentation'].eq('GraphMS v3.5.1 Hybrid + cross-fitted Stage11-v1 + 26-connectivity + no morphology').all():
    raise RuntimeError('Stage12 frozen_segmentation provenance field mismatch')

# Exact scientific column compatibility with the prior audited v2.1 development table.
old_header = pd.read_csv(OLD_STAGE12_DEV, nrows=0).columns.tolist()
if features.columns.tolist() != old_header:
    missing = [c for c in old_header if c not in features.columns]
    extra = [c for c in features.columns if c not in old_header]
    raise RuntimeError(f'Canonical v2.1 table schema drift. Missing={missing}; extra={extra}')

missing_contract = [c for c in required_imaging_features if c not in features.columns]
if missing_contract:
    raise RuntimeError('FAIL CLOSED: Stage13 imaging contract missing columns: ' + ', '.join(missing_contract))

guide_required = [
    'lesion_volume_mm3','lesion_count','guide_max_slice_area_mm2',
    'guide_max_slice_perimeter_mm','guide_max_slice_compactness'
]
missing_guide = [c for c in guide_required if c not in features.columns]
glcm_cols = [c for c in features.columns if '_glcm_' in c]
if missing_guide or not glcm_cols:
    raise RuntimeError(f'Guide Stage12 feature-family gate failed. Missing={missing_guide}; glcm_columns={len(glcm_cols)}')
for mod in ['flair','t1','t2']:
    if not any(c.startswith(mod + '_') for c in features.columns):
        raise RuntimeError(f'Missing canonical multimodal feature family: {mod}')

feature_path = OUT_ROOT / 'stage12_case_features_final_hybrid_93.csv'
lesion_path = OUT_ROOT / 'stage12_lesion_components_final_hybrid_93.csv'
features.to_csv(feature_path, index=False)
lesions.to_csv(lesion_path, index=False)

print('✅ New Hybrid Stage12 case table:', feature_path)
print('✅ New Hybrid Stage12 lesion table:', lesion_path)
print('✅ Cases / patients:', len(features), '/', features.patient_id.nunique())
print('✅ Guide volume/count/area/perimeter/compactness/GLCM families present')
print('✅ FLAIR/T1/T2 canonical feature extraction present')


In [ ]:
#@title 6. Clinical merge — raw clinical_data.csv + canonical prior-merge cross-check { display-mode: "form" }
def locale_float(series):
    s = series.astype('string').str.strip().replace({'':'<NA>','nan':'<NA>','None':'<NA>'})
    return pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce')

raw_clin = pd.read_csv(CLINICAL_CSV, sep=';', dtype=str, encoding='utf-8-sig')
raw_clin.columns = [str(c).lstrip('\ufeff').strip() for c in raw_clin.columns]
required_raw = {'Patient','Timepoint','Age','Sex','MS Type','EDSS','Lesion Number','Lesion Volume'}
if not required_raw.issubset(raw_clin.columns):
    raise RuntimeError(f'clinical_data.csv schema mismatch. Columns={raw_clin.columns.tolist()}')

# The source CSV has a trailing semicolon and, in the audited file, the true lesion-number value is in the trailing unnamed column.
unnamed = [c for c in raw_clin.columns if c.lower().startswith('unnamed')]
lesion_number_col = 'Lesion Number'
if raw_clin['Lesion Number'].fillna('').str.strip().eq('').all() and unnamed:
    nonempty_unnamed = [c for c in unnamed if raw_clin[c].fillna('').str.strip().ne('').any()]
    if len(nonempty_unnamed) != 1:
        raise RuntimeError(f'Cannot unambiguously resolve clinical lesion-number column: {nonempty_unnamed}')
    lesion_number_col = nonempty_unnamed[0]

clinical = pd.DataFrame({
    'patient_id': raw_clin['Patient'].astype(str).str.strip(),
    'timepoint': raw_clin['Timepoint'].astype(str).str.strip(),
    'age': locale_float(raw_clin['Age']),
    'sex': raw_clin['Sex'].astype(str).str.strip(),
    'ms_type': raw_clin['MS Type'].astype(str).str.strip(),
    'target_edss': locale_float(raw_clin['EDSS']),
    'clinical_reported_lesion_volume_qc_only': locale_float(raw_clin['Lesion Volume']),
    'clinical_reported_lesion_number_qc_only': locale_float(raw_clin[lesion_number_col]),
})
clinical['target_edss_ge4'] = (clinical['target_edss'] >= 4.0).astype('Int64')

if clinical[['patient_id','timepoint']].duplicated().any():
    dup = clinical.loc[clinical[['patient_id','timepoint']].duplicated(False), ['patient_id','timepoint']]
    raise RuntimeError('Duplicate patient/timepoint rows in clinical_data.csv:\n' + dup.to_string(index=False))

stage13 = features.merge(clinical, on=['patient_id','timepoint'], how='left', validate='one_to_one')
clinical_cols = ['age','sex','ms_type','target_edss',
                 'clinical_reported_lesion_volume_qc_only',
                 'clinical_reported_lesion_number_qc_only','target_edss_ge4']
if len(stage13) != EXPECTED_CASES or stage13['case_id'].nunique() != EXPECTED_CASES:
    raise RuntimeError('Clinical merge did not preserve exactly 93 unique development cases')
if stage13['patient_id'].nunique() != EXPECTED_PATIENTS:
    raise RuntimeError('Clinical merge patient count is not 53')
if stage13[clinical_cols].isna().any().any():
    miss = stage13.loc[stage13[clinical_cols].isna().any(axis=1), ['case_id'] + clinical_cols]
    raise RuntimeError('Missing clinical values after patient/timepoint-safe merge:\n' + miss.to_string(index=False))

# Fail-closed compatibility check against the old canonical Stage12 package's established clinical merge RESULT ONLY.
# No old imaging or lesion feature values are copied.
old_clin = pd.read_csv(OLD_STAGE13_DEV, usecols=['case_id'] + clinical_cols).sort_values('case_id').reset_index(drop=True)
new_clin = stage13[['case_id'] + clinical_cols].sort_values('case_id').reset_index(drop=True)
if old_clin['case_id'].tolist() != new_clin['case_id'].tolist():
    raise RuntimeError('Canonical prior clinical handoff case IDs differ from refreshed 93-case handoff')
for c in ['age','target_edss','clinical_reported_lesion_volume_qc_only','clinical_reported_lesion_number_qc_only','target_edss_ge4']:
    if not np.allclose(pd.to_numeric(old_clin[c]), pd.to_numeric(new_clin[c]), atol=1e-9, rtol=0, equal_nan=True):
        raise RuntimeError(f'Clinical merge compatibility mismatch in {c}')
for c in ['sex','ms_type']:
    if old_clin[c].astype(str).tolist() != new_clin[c].astype(str).tolist():
        raise RuntimeError(f'Clinical merge compatibility mismatch in {c}')

for c in required_clinical_features + required_targets:
    if c not in stage13.columns:
        raise RuntimeError(f'Stage13 contract column missing after clinical merge: {c}')

stage13_path = OUT_ROOT / 'stage13_DEV_ONLY_93_crosssectional.csv'
stage13.to_csv(stage13_path, index=False)

print('✅ Raw clinical_data.csv parsed directly and merged patient/timepoint-safely')
print('✅ 93 Hybrid-development clinical joins; 53 unique patients')
print('✅ Clinical columns match the prior canonical merge result')
print('✅ OLD Stage12 imaging/lesion feature values were NOT copied')
print('✅ Stage13 development handoff:', stage13_path)


In [ ]:
#@title 7. Preserve the Stage13 feature contract and write the new Hybrid Stage12 manifest { display-mode: "form" }
# Preserve the established feature lists exactly; update only the provenance/governance metadata for this refreshed Hybrid package.
contract_new = copy.deepcopy(contract_ref)
old_imaging = list(contract_ref['recommended_imaging_features'])
old_clinical = list(contract_ref['recommended_clinical_features'])
old_targets = list(contract_ref['targets'])
contract_new['segmentation_source'] = 'GraphMS v3.5.1 Hybrid + cross-fitted Stage11-v1 + 26-connectivity + no morphology'
contract_new['frozen_internal_oof_dice'] = EXPECTED_EQUAL_FOLD_DSC
contract_new['contract_lineage'] = {
    'reference_path': str(OLD_STAGE12_CONTRACT),
    'reference_sha256': sha256_file(OLD_STAGE12_CONTRACT),
    'feature_lists_unchanged': True,
    'provenance_only_refresh': True,
}
contract_new.setdefault('governance', {})['historical_test_rows'] = 'not included in this refreshed Hybrid Stage12 package'
contract_new['governance']['historical_test_included'] = False

if contract_new['recommended_imaging_features'] != old_imaging or contract_new['recommended_clinical_features'] != old_clinical or contract_new['targets'] != old_targets:
    raise RuntimeError('Feature contract lists changed during provenance refresh')
contract_path = OUT_ROOT / 'STAGE13_RECOMMENDED_FEATURE_COLUMNS.json'
atomic_json(contract_path, contract_new)

output_hashes = {
    'stage12_case_features_final_hybrid_93.csv': sha256_file(feature_path),
    'stage12_lesion_components_final_hybrid_93.csv': sha256_file(lesion_path),
    'stage13_DEV_ONLY_93_crosssectional.csv': sha256_file(stage13_path),
    'FINAL_SEGMENTATION_HANDOFF.json': sha256_file(seg_handoff_path),
    'FINAL_HYBRID_MASK_LEDGER.csv': sha256_file(ledger_path),
    'STAGE13_RECOMMENDED_FEATURE_COLUMNS.json': sha256_file(contract_path),
}

manifest = {
    'status':'PENDING_FINAL_AUDIT',
    'stage':12,
    'guide_stage':'Lesion Feature Extraction',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'feature_schema_version':FEATURE_SCHEMA_VERSION,
    'extractor_lineage':'existing audited canonical v2.1 Stage12 engine',
    'segmentation_parent':'GraphMS v3.5.1 Hybrid',
    'protocol_sha':PROTOCOL_SHA,
    'fusion_implementation_id':FUSION_IMPLEMENTATION_ID,
    'inference':'original uniform-overlap Hybrid stored probabilities; no new inference',
    'frozen_equal_fold_dsc':equal_fold_dsc,
    'stage11_cross_fitted_recipes':{str(k):v for k,v in STAGE11_RECIPES.items()},
    'connectivity':26,
    'morphology':'none',
    'ground_truth_mask_used_for_features':False,
    'ground_truth_usage':'segmentation verification only, before Stage12 extraction',
    'n_cases':int(len(features)),
    'n_patients':int(features['patient_id'].nunique()),
    'historical_test_included':False,
    'new_training_performed':False,
    'new_neural_inference_performed':False,
    'atlas_spatial_context':'Harvard-Oxford + periventricular canonical v2.1 context',
    'stage13_feature_contract_satisfied':True,
    'stage13_development_handoff':'stage13_DEV_ONLY_93_crosssectional.csv',
    'important_output_sha256':output_hashes,
}
manifest_path = OUT_ROOT / 'STAGE12_CANONICAL_FEATURE_MANIFEST.json'
atomic_json(manifest_path, manifest)
print('✅ New Hybrid Stage12 manifest prepared (PENDING_FINAL_AUDIT):', manifest_path)


In [ ]:
#@title 8. Final Stage12 hard-gate audit { display-mode: "form" }
def audit_row(gate, passed, details):
    return {'gate':gate, 'status':'PASS' if bool(passed) else 'FAIL', 'details':str(details)}

all_mask_files = sorted(FINAL_MASK_DIR.glob('*.nii.gz'))
pre_rows = [
    audit_row('1. Exact final segmentation DSC reproduced', abs(equal_fold_dsc-EXPECTED_EQUAL_FOLD_DSC) <= DSC_TOL,
              f'{equal_fold_dsc:.12f} expected {EXPECTED_EQUAL_FOLD_DSC:.12f}'),
    audit_row('2. 93/93 final masks', len(all_mask_files)==EXPECTED_CASES and ledger.case.nunique()==EXPECTED_CASES,
              f'{len(all_mask_files)} masks; {ledger.case.nunique()} unique ledger cases'),
    audit_row('3. 93 unique case features', len(features)==EXPECTED_CASES and features.case_id.nunique()==EXPECTED_CASES,
              f'{len(features)} rows; {features.case_id.nunique()} unique cases'),
    audit_row('4. 53 patients', features.patient_id.nunique()==EXPECTED_PATIENTS, features.patient_id.nunique()),
    audit_row('5. GT not used for Stage12 feature extraction', features.ground_truth_mask_used.eq(False).all(), 'all rows false'),
    audit_row('6. No historical ResEncM test branch mixed into refreshed Hybrid package',
              (INCLUDE_HISTORICAL_TEST is False) and set(features.role.unique())=={'development_oof'},
              f'INCLUDE_HISTORICAL_TEST={INCLUDE_HISTORICAL_TEST}; roles={sorted(features.role.unique())}'),
    audit_row('7. lesion volume present', 'lesion_volume_mm3' in features.columns, 'lesion_volume_mm3'),
    audit_row('8. lesion count present', 'lesion_count' in features.columns, 'lesion_count'),
    audit_row('9. area present', 'guide_max_slice_area_mm2' in features.columns, 'guide_max_slice_area_mm2'),
    audit_row('10. perimeter present', 'guide_max_slice_perimeter_mm' in features.columns, 'guide_max_slice_perimeter_mm'),
    audit_row('11. compactness present', 'guide_max_slice_compactness' in features.columns, 'guide_max_slice_compactness'),
    audit_row('12. GLCM texture present', len(glcm_cols)>0, f'{len(glcm_cols)} _glcm_ columns'),
    audit_row('13. FLAIR/T1/T2 available', all(any(c.startswith(m+'_') for c in features.columns) for m in ['flair','t1','t2']),
              'multimodal canonical feature families present'),
    audit_row('14. canonical Stage12 atlas/spatial context available', (not atlas_contract_required) or bool(ATLAS.get('enabled')),
              ATLAS.get('source', ATLAS.get('reason'))),
    audit_row('15. Stage13 feature-column contract satisfied', len(missing_contract)==0, f'{len(required_imaging_features)} imaging contract columns present'),
    audit_row('16. 93 Hybrid-development clinical joins', len(stage13)==EXPECTED_CASES and stage13.case_id.nunique()==EXPECTED_CASES,
              f'{len(stage13)} Hybrid-development clinical joins'),
    audit_row('18. Stage13 development table produced', stage13_path.exists() and len(pd.read_csv(stage13_path))==EXPECTED_CASES,
              stage13_path),
    audit_row('19. Canonical v2.1 table schema preserved', features.columns.tolist()==old_header,
              f'{len(features.columns)} columns match prior audited table header'),
    audit_row('20. Canonical filter5 is idempotent on final Stage11 masks', len(idempotence_failures)==0,
              f'{len(idempotence_failures)} changed masks'),
]
audit_path = OUT_ROOT / 'STAGE12_FINAL_AUDIT.csv'
pre_audit = pd.DataFrame(pre_rows)
if not pre_audit['status'].eq('PASS').all():
    failed_manifest_gate = audit_row('17. manifest complete', False, 'not marked COMPLETE because a prior hard gate failed')
    audit = pd.concat([pre_audit, pd.DataFrame([failed_manifest_gate])], ignore_index=True).sort_values(
        'gate', key=lambda s: s.str.extract(r'^(\d+)')[0].astype(int)
    ).reset_index(drop=True)
    audit.to_csv(audit_path, index=False)
    display(audit)
    raise RuntimeError('FAIL CLOSED: one or more final Stage12 audit gates failed; manifest remains PENDING_FINAL_AUDIT')

# Only after every scientific/data hard gate passes may the package manifest declare COMPLETE.
manifest['status'] = 'COMPLETE'
manifest['final_audit_status'] = 'PASS'
atomic_json(manifest_path, manifest)
manifest_loaded = json.loads(manifest_path.read_text())
manifest_gate = audit_row(
    '17. manifest complete',
    manifest_loaded.get('status')=='COMPLETE' and manifest_loaded.get('n_cases')==EXPECTED_CASES,
    manifest_loaded.get('status')
)
audit = pd.concat([pre_audit, pd.DataFrame([manifest_gate])], ignore_index=True).sort_values(
    'gate', key=lambda s: s.str.extract(r'^(\d+)')[0].astype(int)
).reset_index(drop=True)
audit.to_csv(audit_path, index=False)
display(audit)
if not audit['status'].eq('PASS').all():
    raise RuntimeError('FAIL CLOSED: final manifest gate failed')
print('✅ STAGE12 FINAL AUDIT: PASS')


In [ ]:
#@title FINAL — GraphMS v3.5.1 Hybrid Stage12 completion receipt { display-mode: "form" }
print('='*120)
print('GRAPHMS FINAL HYBRID STAGE12 — COMPLETE')
print('='*120)
print('Training / neural inference       : NO / NO')
print(f'Frozen Hybrid masks               : {len(ledger)} / {EXPECTED_CASES}')
print(f'Frozen Stage11 DSC reproduced     : {equal_fold_dsc:.12f}')
print(f'Feature schema                    : {FEATURE_SCHEMA_VERSION}')
print(f'Cases / patients                  : {len(features)} / {features.patient_id.nunique()}')
print('Ground-truth used for features    : NO')
print('Historical test mixed in          : NO')
print('Guide volume/count/shape/GLCM     : PASS')
print('FLAIR/T1/T2 feature extraction    : PASS')
print('Stage13 feature contract          : PASS')
print('Stage13 development handoff       : READY')
print('Stage12 final audit               : PASS')
print('NEXT                              : REFRESH STAGE13 USING FIXED EXISTING SVM/RIDGE MODEL FAMILIES AND PARAMETERS')
print('='*120)
display(audit)
print('\nOutput namespace:', OUT_ROOT)
